# PACE Speculative Decoding (PARD) — Server Playbook

Interactive walkthrough of **Parallel Draft Model Adaptation (PARD)** on the PACE server:

1. **Launch Commands** — how to start the server with speculative decoding
2. **PARD Streaming Format** — observe multi-token chunks (the signature of spec decode)
3. **Non-Streaming with PARD** — standard request/response (PARD accelerates internally)
4. **Metrics Analysis** — session and Prometheus metrics from the PARD run

---

### How PARD works

A small **draft model** proposes multiple tokens ahead. The main model verifies them in a single forward pass. When the draft is correct, multiple tokens are accepted at once — visible as larger streaming chunks.

### Prerequisites

**Kernel setup** — register the PACE conda environment as a Jupyter kernel (one-time):

```bash
conda activate pace-env-py3.12
pip install ipykernel
python -m ipykernel install --user --name pace-env-py3.12 --display-name "PACE (py3.12)"
```

Then select **PACE (py3.12)** from the kernel picker (top-right corner in Jupyter).

**Start the server** by running the cell below, or in a separate terminal:

```bash
conda activate pace-env-py3.12
pace-server --server_model Qwen/Qwen2.5-7B-Instruct --dtype bfloat16 \
  --spec_config '{"model_name":"amd/PARD-Qwen2.5-0.5B","num_speculative_tokens":12}' \
  --serve_type continuous_prefill_first \
  --scheduler_metrics_enabled True --enable_prometheus
```

## Setup

Imports and configuration. Change `MODEL`, `DRAFT`, or `ROUTER_URL` here if your setup differs.

In [ ]:
# ******************************************************************************
# Copyright (c) 2026 Advanced Micro Devices, Inc.
# All rights reserved.
# Portion of this file may consist of AI-generated code.
# ******************************************************************************

import json
import os
import subprocess
import time

import httpx
from transformers import AutoTokenizer

# Ensure the conda env's binaries (pace-server, etc.) are on PATH
CONDA_PREFIX = os.environ.get("CONDA_PREFIX", "")
if CONDA_PREFIX:
    conda_bin = os.path.join(CONDA_PREFIX, "bin")
    if conda_bin not in os.environ["PATH"]:
        os.environ["PATH"] = conda_bin + ":" + os.environ["PATH"]

MODEL = "Qwen/Qwen2.5-7B-Instruct"
DRAFT = "amd/PARD-Qwen2.5-0.5B"
ROUTER_URL = "http://localhost:8080"

SPEC_CFG = json.dumps({"model_name": DRAFT, "num_speculative_tokens": 12})
LAUNCH_CMD = (
    f"pace-server --server_model {MODEL} --dtype bfloat16 \\\n"
    f"  --spec_config '{SPEC_CFG}' \\\n"
    f"  --serve_type continuous_prefill_first \\\n"
    f"  --scheduler_metrics_enabled True --enable_prometheus"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL)

print(f"Model : {MODEL}")
print(f"Draft : {DRAFT}")
print(f"Router: {ROUTER_URL}")
print(f"Tokenizer: {tokenizer.__class__.__name__} ({tokenizer.vocab_size} vocab)")
print(
    f"PATH includes: {CONDA_PREFIX}/bin"
    if CONDA_PREFIX
    else "Warning: CONDA_PREFIX not set"
)

## Start Server (optional)

Run this cell to launch the PARD-enabled server **in the background** from within the notebook.

Skip this cell if you already have a server running in a separate terminal.

In [ ]:
CONDA_ENV = "pace-env-py3.12"

server_cmd = (
    f"conda run --no-capture-output -n {CONDA_ENV} "
    f"pace-server --server_model {MODEL} --dtype bfloat16 "
    f"--spec_config '{SPEC_CFG}' "
    f"--serve_type continuous_prefill_first "
    f"--scheduler_metrics_enabled True --enable_prometheus"
)

server_proc = subprocess.Popen(
    server_cmd,
    shell=True,
    stdout=open("pace_server_pard.log", "w"),
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
print(f"Server starting (PID {server_proc.pid})...")
print(f"Command: {server_cmd}")
print(f"Logs:    pace_server_pard.log")
print("\nWait ~30-60s for model loading, then run the Server Check cell below.")

## Server Check

Verify the server is reachable, the scheduler is running, and the expected model is loaded.

If the check fails, the cell prints the exact launch command you need.

In [ ]:
async def check_server():
    try:
        async with httpx.AsyncClient(timeout=10.0) as client:
            resp = await client.get(f"{ROUTER_URL}/v1/health")
            health = resp.json()
    except Exception as e:
        print(f"Cannot reach server at {ROUTER_URL}: {e}")
        print(f"\nStart the server with:\n\n    {LAUNCH_CMD}\n")
        return False

    if not health.get("scheduler_running"):
        print(f"Scheduler not running.\n\n    {LAUNCH_CMD}\n")
        return False

    try:
        async with httpx.AsyncClient(timeout=60.0) as client:
            probe = await client.post(
                f"{ROUTER_URL}/v1/completions",
                json={
                    "model": MODEL,
                    "prompt": "probe",
                    "stream": False,
                    "max_tokens": 1,
                },
            )
            if (
                probe.status_code == 404
                and probe.json().get("error", {}).get("code") == "model_not_found"
            ):
                print(f"Model '{MODEL}' not loaded on server.")
                print(f"\nStart the server with:\n\n    {LAUNCH_CMD}\n")
                return False
    except Exception:
        pass

    print(f"Server healthy (queue={health.get('queue_size', 0)})")
    return True


assert await check_server(), "Server not ready — see message above."

## 1. Launch Commands

Reference for how to start the server with PARD.

### Supported model / draft pairs

| Target Model | Draft Model |
|---|---|
| `Qwen/Qwen2.5-7B-Instruct` | `amd/PARD-Qwen2.5-0.5B` |
| `Qwen/Qwen2.5-14B-Instruct` | `amd/PARD-Qwen2.5-0.5B` |
| `Qwen/Qwen2.5-3B-Instruct` | `amd/PARD-Qwen2.5-0.5B` |
| `meta-llama/Llama-3.1-8B-Instruct` | `amd/PARD-Llama-68M` |

### Single instance

```bash
pace-server --server_model Qwen/Qwen2.5-7B-Instruct --dtype bfloat16 \
  --spec_config '{"model_name":"amd/PARD-Qwen2.5-0.5B","num_speculative_tokens":12}' \
  --serve_type continuous_prefill_first \
  --scheduler_metrics_enabled True --enable_prometheus
```

### Multi-instance with monitoring

```bash
pace-server --server_model Qwen/Qwen2.5-7B-Instruct --dtype bfloat16 \
  --spec_config '{"model_name":"amd/PARD-Qwen2.5-0.5B","num_speculative_tokens":12}' \
  --num_engine_instances 2 \
  --serve_type continuous_prefill_first \
  --scheduler_metrics_enabled True --enable_prometheus
```

## 2. PARD Streaming

With PARD, each streaming chunk can carry **multiple tokens** — the draft model proposes several, and the verifier accepts them in bulk. This section streams output live and uses the tokenizer to count the exact number of tokens in each chunk.

After each prompt, a summary shows total tokens, decode steps, and **mean accepted tokens per step** — the key PARD efficiency metric.

In [ ]:
prompts = [
    (
        "Factual",
        "The three states of matter are solid, liquid, and gas. Explain each state and give an example:",
    ),
    ("Creative", "Write a limerick about a programmer who debugs at night:"),
    (
        "Code",
        "Write a Python function to compute the Fibonacci sequence up to n terms:",
    ),
]

async with httpx.AsyncClient(timeout=300.0) as client:
    for label, prompt in prompts:
        print(f"\n{'=' * 60}")
        print(f"[{label}]  {prompt[:70]}{'...' if len(prompt) > 70 else ''}")
        print(f"{'=' * 60}\n")

        payload = {
            "model": MODEL,
            "prompt": prompt,
            "stream": True,
            "max_tokens": 150,
            "temperature": 0,
        }
        hdrs = {"Content-Type": "application/json", "Accept": "text/event-stream"}

        full_text = ""
        tokens_per_step = []
        t0 = time.time()
        first_token = None

        try:
            async with client.stream(
                "POST", f"{ROUTER_URL}/v1/completions", json=payload, headers=hdrs
            ) as resp:
                buf = ""
                async for raw in resp.aiter_text():
                    buf += raw
                    while "\n" in buf:
                        line, buf = buf.split("\n", 1)
                        line = line.strip()
                        if not line or not line.startswith("data: "):
                            continue
                        if line == "data: [DONE]":
                            break
                        try:
                            obj = json.loads(line[6:])
                            content = obj.get("choices", [{}])[0].get("text", "")
                            if content:
                                if first_token is None:
                                    first_token = time.time()
                                n_tok = len(
                                    tokenizer.encode(content, add_special_tokens=False)
                                )
                                tokens_per_step.append(n_tok)
                                full_text += content
                                print(content, end="", flush=True)
                        except json.JSONDecodeError:
                            continue
        except Exception as e:
            print(f"\n  Request failed: {e}")
            continue

        elapsed = time.time() - t0
        ttft = (first_token - t0) if first_token else elapsed
        total_tok = len(tokenizer.encode(full_text, add_special_tokens=False))
        steps = len(tokens_per_step)
        mean_accepted = total_tok / steps if steps else 0

        print(
            f"\n\n  Tokens: {total_tok}  |  Steps: {steps}  |  "
            f"Mean accepted/step: {mean_accepted:.1f}"
        )
        print(
            f"  TTFT: {ttft:.3f}s  |  Total: {elapsed:.2f}s  |  "
            f"{total_tok / elapsed:.1f} tok/s"
        )

## 3. Non-Streaming with PARD

Non-streaming mode returns the complete response after all tokens are generated. PARD accelerates generation internally — the response format is **identical** to a non-speculative server.

This means you can enable PARD on the server side without changing any client code.

In [ ]:
prompts = [
    "List the planets in our solar system in order from the sun.",
    "What is the Pythagorean theorem? State it and give a brief proof.",
]

async with httpx.AsyncClient(timeout=300.0) as client:
    for prompt in prompts:
        payload = {
            "model": MODEL,
            "prompt": prompt,
            "stream": False,
            "max_tokens": 150,
            "temperature": 0,
        }
        t0 = time.time()
        resp = await client.post(f"{ROUTER_URL}/v1/completions", json=payload)
        elapsed = time.time() - t0
        result = resp.json()

        text = result.get("choices", [{}])[0].get("text", "N/A")
        print(f"Prompt: {prompt}")
        print(f"Output: {text[:200]}")
        print(f"Time:   {elapsed:.2f}s  |  ID: {result.get('id', 'N/A')}")
        print()

## 4. PARD Metrics Analysis

Two metrics sources are available when the server is started with `--scheduler_metrics_enabled True --enable_prometheus`:

| Source | Endpoint | What it provides |
|---|---|---|
| **Session metrics** | `GET /v1/server_metrics` | Per-scheduler-session aggregates (JSON) |
| **Prometheus** | `GET /metrics/` | Standard Prometheus exposition format with TTFT/TPOT histograms |

This cell fetches both and computes average TTFT and TPOT from the Prometheus counters.

In [ ]:
async with httpx.AsyncClient(timeout=30.0) as client:
    # Session metrics
    print("Scheduler session metrics (GET /v1/server_metrics):")
    try:
        resp = await client.get(f"{ROUTER_URL}/v1/server_metrics")
        for k, v in resp.json().items():
            print(f"  {k}: {v}")
    except Exception:
        print("  Unavailable (requires --scheduler_metrics_enabled True)")

    # Prometheus
    print("\nPrometheus TTFT / TPOT summary (GET /metrics/):")
    try:
        resp = await client.get(f"{ROUTER_URL}/metrics/")
        text = resp.text
        for line in text.split("\n"):
            if line.startswith("pace_") and ("_sum" in line or "_count" in line):
                print(f"  {line}")

        def _avg(prefix):
            cl = [l for l in text.split("\n") if l.startswith(f"{prefix}_count")]
            sl = [l for l in text.split("\n") if l.startswith(f"{prefix}_sum")]
            if cl and sl:
                c, s = float(cl[0].split()[-1]), float(sl[0].split()[-1])
                if c > 0:
                    return s / c, int(c)
            return None, 0

        print()
        avg_ttft, n = _avg("pace_ttft_seconds")
        if avg_ttft is not None:
            print(f"  Avg TTFT: {avg_ttft:.4f}s  ({n} requests)")
        avg_tpot, n = _avg("pace_tpot_seconds")
        if avg_tpot is not None:
            print(f"  Avg TPOT: {avg_tpot:.4f}s  ({n} requests)")
    except Exception:
        print("  Unavailable (requires --enable_prometheus)")

## Teardown

Run the cell below to stop the server that was launched from this notebook. If you started the server in a separate terminal, press **Ctrl+C** there instead.

The server is **stateless** — restarting is safe at any time.

In [ ]:
import os, signal, subprocess as _sp

if "server_proc" in dir() and server_proc.poll() is None:
    pid = server_proc.pid
    print(f"Stopping server (PID {pid})...")
    try:
        pgid = os.getpgid(pid)
        os.killpg(pgid, signal.SIGTERM)
    except (ProcessLookupError, PermissionError):
        server_proc.terminate()
    try:
        server_proc.wait(timeout=20)
        print("Server stopped.")
    except _sp.TimeoutExpired:
        print("Graceful stop timed out, force killing...")
        try:
            os.killpg(os.getpgid(pid), signal.SIGKILL)
        except (ProcessLookupError, PermissionError):
            server_proc.kill()
        server_proc.wait(timeout=5)
        print("Server killed.")
else:
    print("No server process found (was it started from a terminal instead?)")

print("\nOptional cleanup:  rm -rf ~/.cache/pace/prometheus/")